# 01. Подготовка данных и EDA

Ноутбук закрывает три задачи этапа 1:

| № | Задача | Результат |
|---|---|---|
| 1 | Сбор и подготовка данных: получение KazNERD и UD Kazakh Treebank | Подготовленный датасет, описание структуры данных |
| 2 | Первичная очистка, единый формат, разбиения, фиксация версий | Визуализация данных до и после предобработки |
| 3 | Исследовательский анализ: выбросы, пропуски, классы, длины, редкие слова, дисбаланс | Аналитический отчёт с графиками |

Весь код лежит в `src/data/`; ноутбук только вызывает функции и показывает результат.
Рисунки сохраняются на Drive в `thesis/outputs/figures/` (PNG, 300 dpi — для диссертации).

## 0. Настройка окружения

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    token = userdata.get('GH_TOKEN')
    !git clone -q https://{token}@github.com/zhannaa0/kazakh-hybrid-nlp.git /content/project 2>/dev/null || git -C /content/project pull -q
    %cd /content/project
    !pip install -q -r requirements.txt
    PROJECT = Path('/content/project')
    THESIS = Path('/content/drive/MyDrive/thesis')
else:                                   # запуск на своём компьютере
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    THESIS = Path(os.environ.get('THESIS_DIR', PROJECT.parent / 'thesis'))

sys.path.insert(0, str(PROJECT))
RAW  = THESIS / 'data'                  # исходные файлы: kaznerd/, ud-ktb/
PROC = THESIS / 'data' / 'processed'    # результат подготовки
FIG  = THESIS / 'outputs' / 'figures'   # рисунки для диссертации
print(RAW, PROC, FIG, sep='\n')

In [ ]:
import json
import pandas as pd
from IPython.display import display

from src.data import eda, plots
from src.data.prepare import load_raw, load_processed, prepare, read_jsonl

plots.style()
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_colwidth', 80)

## 1. Сбор и подготовка данных

**Источники** (скачаны с GitHub и лежат на Drive в `thesis/data/`):

| Датасет | Задачи | Источник | Формат |
|---|---|---|---|
| KazNERD | NER (25 типов сущностей) | github.com/IS2AI/KazNERD | IOB2: «токен ТЕГ», пустая строка между предложениями |
| UD Kazakh-KTB, релиз r2.18 | POS (UPOS), морфология (FEATS) | github.com/UniversalDependencies/UD_Kazakh-KTB | CoNLL-U: 10 колонок через табуляцию |

Сначала читаем файлы **как есть**, без исправлений — это состояние «до».

In [ ]:
for f in sorted((RAW / 'kaznerd').glob('*.txt')) + sorted((RAW / 'ud-ktb').glob('*.conllu')):
    print(f'{f.parent.name}/{f.name:28} {f.stat().st_size / 1e6:6.1f} MB')

kaz_raw, ud_raw, ud_format = load_raw(RAW)
print('\nМногословные токены CoNLL-U (пропущены, работаем с синтаксическими словами):', ud_format)

**Как выглядит исходный файл.** KazNERD — по токену на строку; UD-KTB — 10 колонок,
из которых нам нужны FORM (токен), LEMMA, UPOS (часть речи) и FEATS (морфологические признаки).
Строки вида `12-13 Баласыз` — многословные токены: слово «Баласыз» в UD разбито на
синтаксические слова «бала» + «сыз». Разметка UPOS/FEATS стоит на синтаксических словах,
поэтому работаем с ними.

In [ ]:
print(''.join(open(RAW / 'kaznerd' / 'IOB2_train.txt', encoding='utf-8').readlines()[:10]))
print(''.join(open(RAW / 'ud-ktb' / 'kk_ktb-ud-train.conllu', encoding='utf-8').readlines()[:16]))

### Структура данных в едином формате

После чтения оба датасета — это списки предложений-словарей с общими полями
`id` и `tokens` и полями разметки своей задачи. Тот же формат сохраняется в JSONL
(одна строка = одно предложение).

In [ ]:
k = kaz_raw['train'][0]
u = ud_raw['train'][0]
print(json.dumps(k, ensure_ascii=False)[:300], '...\n')
print(json.dumps({key: (v[:6] if isinstance(v, list) else v) for key, v in u.items()}, ensure_ascii=False, indent=1))

In [ ]:
structure = pd.DataFrame([
    ('id', 'оба', 'str', 'уникальный идентификатор предложения'),
    ('tokens', 'оба', 'list[str]', 'токены (для UD — синтаксические слова)'),
    ('ner', 'KazNERD', 'list[str]', 'теги IOB2: B-ТИП / I-ТИП / O, 25 типов'),
    ('doc', 'UD-KTB', 'str', 'исходный текст (19 текстов) — для стратификации'),
    ('text', 'UD-KTB', 'str', 'предложение строкой'),
    ('lemmas', 'UD-KTB', 'list[str]', 'леммы'),
    ('upos', 'UD-KTB', 'list[str]', 'универсальные части речи, 17 тегов'),
    ('feats', 'UD-KTB', 'list[str]', 'признаки «Case=Nom|Number=Sing», «_» — признаков нет'),
], columns=['поле', 'датасет', 'тип', 'описание'])
structure

## 2. Очистка, единый формат, разбиения, фиксация версий

`prepare()` делает всё за один вызов (код — `src/data/prepare.py`):

1. **Очистка** (`clean.py`), одинаково для всех сплитов:
   - Unicode NFC;
   - латинские буквы-двойники внутри кириллических слов → кириллица (`Aстана` → `Астана`, `мемлекеттiк` → `мемлекеттік`);
   - KazNERD: `I-X` без начала сущности → `B-X`.
2. **Разбиения** (`splits.py`):
   - KazNERD — официальный сплит без изменений (сравнимость со статьёй KazNERD);
   - UD-KTB — официальный train всего 31 предложение, поэтому train+test объединяются
     и делятся заново 80/10/10, **стратифицированно по исходному тексту**, seed = 42.
3. **Фиксация версий** — `manifest.json`: SHA-256 исходных и выходных файлов, версия UD,
   seed, версия предобработки, коммит кода. Если хэш исходника изменится — будет видно сразу.

Предложения не удаляются: дубликаты и утечки только измеряются.

In [ ]:
manifest = prepare(RAW, PROC, seed=42, ud_version='r2.18')
pd.DataFrame({ds: {sp: c['sentences'] for sp, c in m.items()} for ds, m in manifest['counts'].items()})

In [ ]:
print(json.dumps({k: manifest[k] for k in ('created_at', 'preprocessing_version', 'code_commit', 'params')}, indent=1))
print(json.dumps(manifest['sources'], indent=1, ensure_ascii=False))

### Что изменила очистка

In [ ]:
log = pd.DataFrame(read_jsonl(PROC / 'cleaning_log.jsonl'))
log['dataset'] = log['id'].str.startswith('kaznerd').map({True: 'KazNERD', False: 'UD-KTB'})
display(log.groupby(['dataset', 'rule', 'field']).size().rename('corrections').to_frame())
log.sample(10, random_state=0)[['dataset', 'rule', 'before', 'after']]

In [ ]:
fig = plots.cleaning_summary(log['rule'].value_counts().rename({
    'homoglyph': 'Latin look-alike letters → Cyrillic',
    'iob2_repair': 'Invalid I- tag → B- tag',
    'unicode_nfc': 'Unicode NFC normalisation'}))
plots.save(fig, '01_cleaning_corrections', FIG)

In [ ]:
kaz = load_processed(PROC, 'kaznerd')
ud  = load_processed(PROC, 'udktb')
eda.ner_quality_summary(kaz_raw, kaz)

### Разбиения до и после

In [ ]:
sizes = eda.split_sizes({'KazNERD': kaz, 'UD-KTB official': ud_raw, 'UD-KTB re-split': ud})
display(sizes)
fig = plots.split_sizes(sizes)
plots.save(fig, '02_split_sizes', FIG)

Официальный train UD-KTB не просто мал — он целиком состоит из **других** текстов, чем test.
Новый сплит повторяет жанровый состав во всех трёх частях:

In [ ]:
before, after = eda.ud_domain_mix(ud_raw), eda.ud_domain_mix(ud)
mixes = {'Official train': before['train'], 'Official test': before['test'],
         'Re-split train': after['train'], 'Re-split dev': after['dev'], 'Re-split test': after['test']}
display(pd.DataFrame(mixes))
fig = plots.domain_mix(mixes)
plots.save(fig, '03_ud_domain_mix', FIG)

## 3. Исследовательский анализ данных

Дальше всё считается на **очищенных** данных с итоговыми сплитами.

### 3.1. Распределение классов сущностей (KazNERD)

In [ ]:
ner_counts = eda.ner_class_counts(kaz)
display(ner_counts)
display(eda.imbalance_summary(ner_counts['train']).to_frame('KazNERD train'))
fig = plots.class_distribution(ner_counts['train'], 'KazNERD: entities per type (train)',
                               'Entities (log scale)', log=True)
plots.save(fig, '04_ner_class_distribution', FIG)

Согласованы ли сплиты: доля каждого типа в train, dev и test.

In [ ]:
shares = eda.split_consistency(ner_counts)
print(f"Максимальное расхождение долей между сплитами: {shares['max_abs_diff_pp'].max():.2f} п.п.")
fig = plots.split_shares(shares, 'KazNERD: entity type shares by split')
plots.save(fig, '05_ner_split_consistency', FIG)

### 3.2. Распределение частей речи (UD-KTB)

In [ ]:
upos_counts = eda.token_label_counts(ud, 'upos')
display(upos_counts)
display(eda.imbalance_summary(upos_counts['train']).to_frame('UD-KTB train'))
fig = plots.class_distribution(upos_counts['train'], 'UD-KTB: UPOS tags (train)', 'Words')
plots.save(fig, '06_upos_distribution', FIG)

### 3.3. Морфологические признаки (UD-KTB)

In [ ]:
feats = eda.feature_counts(ud)
fig = plots.class_distribution(feats, 'UD-KTB: morphological categories (train)', 'Words carrying the feature')
plots.save(fig, '07_morph_features', FIG)
eda.feats_label_space(ud).to_frame('value')

`distinct_feats_strings` — сколько разных полных наборов признаков: это число классов, если
предсказывать FEATS целиком. Треть из них встречается в train один раз — аргумент в пользу
предсказания признаков по отдельности и в пользу правил.

### 3.4. Длина предложений и выбросы

In [ ]:
lengths = eda.sentence_lengths({'KazNERD': kaz, 'UD-KTB': ud})
outl = eda.length_outliers(lengths)
display(outl)
fig = plots.sentence_lengths(lengths, outl)
plots.save(fig, '08_sentence_lengths', FIG)

In [ ]:
# Самые длинные предложения — посмотреть глазами, не мусор ли это
longest = sorted(kaz['train'], key=lambda s: len(s['tokens']), reverse=True)[:3]
for s in longest:
    print(len(s['tokens']), ' '.join(s['tokens'])[:250], '...\n')

In [ ]:
display(eda.token_outliers(kaz).assign(dataset='KazNERD'))
eda.token_outliers(ud).assign(dataset='UD-KTB')

### 3.5. Редкие слова и OOV

In [ ]:
vocab = {'KazNERD': eda.vocabulary_stats(kaz), 'UD-KTB': eda.vocabulary_stats(ud)}
for name, v in vocab.items():
    print(name); display(v)
buckets = {'KazNERD': eda.frequency_buckets(kaz), 'UD-KTB': eda.frequency_buckets(ud)}
fig = plots.frequency_buckets(buckets)
plots.save(fig, '09_word_frequency', FIG)
fig = plots.oov_rates(vocab)
plots.save(fig, '10_oov_rate', FIG)

### 3.6. Дубликаты, утечки и пропуски в разметке

In [ ]:
pd.DataFrame({ds: {'duplicates in train': q['duplicates_within']['train'],
                   'dev sentences also in train': q['leakage_from_train']['dev']['n'],
                   'test sentences also in train': q['leakage_from_train']['test']['n'],
                   'test leakage share': f"{q['leakage_from_train']['test']['share']:.1%}"}
              for ds, q in manifest['quality'].items()})

**Кандидаты в пропуски разметки NER**: одна и та же строка размечена как сущность
в одних предложениях и не размечена в других. Это кандидаты для ручной проверки,
не доказанные ошибки.

In [ ]:
gaps = eda.ner_annotation_gaps(kaz)
print('Кандидатов:', len(gaps))
gaps.head(15)

In [ ]:
# UD-KTB: пустые леммы/UPOS и знаменательные слова без признаков
eda.ud_annotation_gaps(ud)

## Выводы

1. **Масштаб.** KazNERD — 112 702 предложения (≈1,3 млн токенов), UD-KTB — 1 078 предложений
   (≈10,5 тыс. слов): разница на два порядка. Это задуманный контраст «больше ресурсов / мало ресурсов».
2. **Качество исходных данных высокое.** Ошибок формата нет; исправлено 76 латинских букв-двойников
   и 16 некорректных тегов IOB2 (только KazNERD train). Предложения не удалялись.
3. **Сплит UD-KTB пришлось пересобрать.** Официальный train — 31 предложение и целиком из других
   текстов, чем test. Новый сплит 864 / 107 / 107 стратифицирован по источнику: жанровый состав
   всех частей одинаков.
4. **Сильный дисбаланс классов.** KazNERD: CARDINAL, DATE и GPE — около половины всех сущностей,
   NON_HUMAN — 6 примеров в train (дисбаланс ≈ 3900:1), 8 типов < 1%. Поэтому кроме micro-F1
   нужно смотреть macro-F1 и F1 по классам. Доли классов в сплитах KazNERD согласованы (< 1 п.п.).
5. **Редкие слова — главная трудность UD-KTB.** 41–45% словоупотреблений dev/test не встречаются в train
   (KazNERD — ~5%). Для агглютинативного казахского это ожидаемо и прямо мотивирует
   морфологические правила в гибриде.
6. **Морфология.** 319 разных наборов FEATS, треть — единичные; почти все признаки в dev/test
   встречались в train (неизвестных наборов ~1,6%).
7. **Утечка и шум разметки KazNERD:** ~2,7% предложений test дословно есть в train, 19 предложений
   размечены противоречиво, найдены кандидаты в пропуски сущностей. Официальный сплит оставлен
   ради сравнимости; эти цифры стоит указать в главе о данных.